<a href="https://colab.research.google.com/github/kexincchen/NLP-Project/blob/main/GroupID__COMP90042_Project_2024.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2024 COMP90042 Project
*Make sure you change the file name with your group id.*

# Readme
*If there is something to be noted for the marker, please mention here.*

*If you are planning to implement a program with Object Oriented Programming style, please put those the bottom of this ipynb file*

# 1.DataSet Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [1]:
import json
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
import nltk

# Ensure necessary NLTK downloads
nltk.download('punkt')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    """Preprocesses text by lowercasing, tokenizing, removing stopwords and stemming."""
    tokens = word_tokenize(text.lower())
    filtered_tokens = [stemmer.stem(word) for word in tokens if word.isalpha() and word not in stop_words]
    return " ".join(filtered_tokens)

EVIDENCE_FILE = 'data/evidence.json'
TRAIN_FILE = 'data/train-claims.json'

def load_data(file_path):
    """Loads data from a JSON file."""
    with open(file_path, 'r') as file:
        data = json.load(file)
    return data

evidence_data = load_data(EVIDENCE_FILE)
claims_data = load_data(TRAIN_FILE)

# Map evidence IDs to their texts
# evidence_map = {eid: preprocess_text(text) for eid, text in evidence_data.items()}


[nltk_data] Downloading package punkt to /Users/clarec/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/clarec/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


KeyboardInterrupt: 

In [4]:
import json

# Assuming evidence_map is already created and filled
def save_to_json(filepath, data):
    """ Save a dictionary to a JSON file. """
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

# Example usage
save_to_json('data/curated/preprocessed_evidence_map.json', evidence_map)

In [ ]:
def load_from_json(filepath):
    """ Load a dictionary from a JSON file. """
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

# Example usage
evidence_map = load_from_json('data/curated/preprocessed_evidence_map.json')


In [3]:
data_for_dataframe = []
for claim_id, claim_details in claims_data.items():
    claim_text = preprocess_text(claim_details['claim_text'])
    # for eid, text in evidence_map.items():
    #     label = 0
    #     if eid in claim_details['evidences']:
    #         label = 1
    #     data_for_dataframe.append({
    #         'claim_text': claim_text,
    #         'evidence_text': text,
    #         'label': label  
    #     })  
    for eid in claim_details['evidences']:
        evidence_text = evidence_map.get(eid, "NULL")  
        if evidence_text == "NULL":
            continue
        data_for_dataframe.append({
            'claim': claim_text,
            'evidence': evidence_text,
            'label': 1  # As all evidences are relevant
        })

# Create DataFrame
claims_df = pd.DataFrame(data_for_dataframe)

# Vectorization
vectorizer = TfidfVectorizer()
all_texts = claims_df['claim'].tolist() + claims_df['evidence'].tolist()

vectorizer.fit(all_texts)  # Fit the vectorizer on both claims and evidences
claims_df['claim'] = list(vectorizer.transform(claims_df['claim']).toarray())
claims_df['evidence'] = list(vectorizer.transform(claims_df['evidence']).toarray())

# Display the DataFrame to verify
print(claims_df.head())

                                               claim  \
0  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
1  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
2  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
3  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
4  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   

                                            evidence  label  
0  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...      1  
1  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...      1  
2  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...      1  
3  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...      1  
4  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...      1  


In [5]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import random


data_for_dataframe = []
evidence_keys = list(evidence_map.keys())  # List of all evidence IDs

for claim_id, claim_details in claims_data.items():
    claim_text = preprocess_text(claim_details['claim_text'])
    claim_evidences = set(claim_details['evidences'])  # Convert to set for faster checks

    # Add positive examples
    for eid in claim_evidences:
        evidence_text = evidence_map.get(eid, "NULL")  
        if evidence_text != "NULL":
            data_for_dataframe.append({
                'claim': claim_text,
                'evidence': evidence_text,
                'label': 1  # Label as relevant
            })

    # Add negative examples
    num_neg_samples = min(len(claim_evidences), len(evidence_keys) - len(claim_evidences))  # Limit the number of negative samples
    negative_samples = random.sample([k for k in evidence_keys if k not in claim_evidences], num_neg_samples)
    for eid in negative_samples:
        evidence_text = evidence_map[eid]
        data_for_dataframe.append({
            'claim': claim_text,
            'evidence': evidence_text,
            'label': 0  # Label as not relevant
        })

# Create DataFrame
claims_df = pd.DataFrame(data_for_dataframe)

# Vectorization should be done after DataFrame creation to ensure data integrity
vectorizer = TfidfVectorizer()

# It's important to vectorize the text data for the machine learning model
claims_df['claim'] = list(vectorizer.fit_transform(claims_df['claim']).toarray())
claims_df['evidence'] = list(vectorizer.transform(claims_df['evidence']).toarray())

# Display the DataFrame to verify
print(claims_df.head())


                                               claim  \
0  scientif evid pollut higher concentr actual he...   
1  scientif evid pollut higher concentr actual he...   
2  scientif evid pollut higher concentr actual he...   
3  scientif evid pollut higher concentr actual he...   
4  scientif evid pollut higher concentr actual he...   

                                            evidence  label  \
0  high concentr time atmospher concentr greater ...      1   
1  higher carbon dioxid concentr favour affect pl...      1   
2  plant grow much percent faster concentr ppm co...      1   
3  structur beta subunit determin found fold arou...      0   
4  sweden repres inger berggren sing sol och vår ...      0   

                                        claim_vector  \
0  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
1  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
2  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
3  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...   
4  [

# 2. Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [6]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Extract feature arrays for the classifier
# Since we stored the vectors as lists, we need to convert them back to numpy arrays
X_claim_vectors = np.array(list(claims_df['claim_vector']))
X_evidence_vectors = np.array(list(claims_df['evidence_vector']))

# Combine the vectors into a single feature array for each sample
X_combined = np.hstack((X_claim_vectors, X_evidence_vectors))
y = claims_df['label'].values

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.2, random_state=42)

# Initialize and train the logistic regression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Predict on the testing set
y_pred = model.predict(X_test)

# Evaluate the model
print("Classification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.94      0.91       832
           1       0.93      0.87      0.90       817

    accuracy                           0.90      1649
   macro avg       0.91      0.90      0.90      1649
weighted avg       0.91      0.90      0.90      1649



# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [7]:
# Function to predict if evidence supports the claim
def predict_support(model, vectorizer, claim, evidences):
    # Preprocess texts
    claim_processed = preprocess_text(claim)
    evidences_processed = [preprocess_text(evidence) for evidence in evidences]

    # Vectorize texts
    texts = [claim_processed] + evidences_processed
    text_vectors = vectorizer.transform(texts)

    # The first vector corresponds to the claim, the rest to the evidences
    claim_vector = text_vectors[0]
    evidence_vectors = text_vectors[1:]

    # Make predictions for each evidence
    results = []
    for evidence_vector in evidence_vectors:
        combined_vector = np.hstack([claim_vector.toarray(), evidence_vector.toarray()])
        prediction = model.predict(combined_vector.reshape(1, -1))
        results.append(prediction[0])

    return results

# Example data
claim = "Global warming trends correlate with increased CO2 levels."
evidences = [
    "CO2 levels have been rising alarmingly.",
    "Urban areas have seen a decrease in air quality.",
    "Climate models demonstrate a strong correlation between CO2 concentrations and temperature increases."
]

# Predict if evidences support the claim
support_predictions = predict_support(model, vectorizer, claim, evidences)

# Output results
for evidence, supported in zip(evidences, support_predictions):
    print(f"Evidence: {evidence}\nSupports Claim: {'Yes' if supported == 1 else 'No'}\n")

Evidence: CO2 levels have been rising alarmingly.
Supports Claim: Yes

Evidence: Urban areas have seen a decrease in air quality.
Supports Claim: No

Evidence: Climate models demonstrate a strong correlation between CO2 concentrations and temperature increases.
Supports Claim: Yes



## Object Oriented Programming codes here

*You can use multiple code snippets. Just add more if needed*

In [2]:
import json
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from gensim.models import KeyedVectors

# Load JSON data
def load_data(filepath):
    with open(filepath, 'r') as file:
        data = json.load(file)
    return data

# Load claims and evidence
claims_data = load_data('data/train-claims.json')
evidence_data = load_data('data/evidence.json')

# Example data format adjustment
# Let's assume claims_data and evidence_data are dictionaries where keys are IDs and values are text
claims = [info['claim_text'] for info in claims_data.values()]
evidences = [evidence_data[eid] for claim in claims_data.values() for eid in claim['evidences']]



In [4]:
# Load pre-trained word2vec
word_vectors = KeyedVectors.load('word2vec.wordvectors', mmap='r')

# Tokenize text data
tokenizer = Tokenizer()
tokenizer.fit_on_texts(claims + evidences)
vocab_size = len(tokenizer.word_index) + 1

# Convert texts to sequences of integers
claims_seq = tokenizer.texts_to_sequences(claims)
evidences_seq = tokenizer.texts_to_sequences(evidences)

# Pad sequences to ensure uniform length
max_length = max(max(len(seq) for seq in claims_seq), max(len(seq) for seq in evidences_seq))
claims_padded = pad_sequences(claims_seq, maxlen=max_length, padding='post')
evidences_padded = pad_sequences(evidences_seq, maxlen=max_length, padding='post')

###  Creating the Embedding Matrix
Create an embedding matrix that will be used to weight the embedding layer in the Keras model.

In [6]:
# Create an embedding matrix
embedding_dim = 50  # dimension of word2vec vectors
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in tokenizer.word_index.items():
    if word in word_vectors:
        embedding_vector = word_vectors[word]
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector


### Building the LSTM Model

We will build a simple unidirectional LSTM model to compare claim and evidence embeddings.

In [7]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Dropout, concatenate

# Define the model
def create_model():
    # Inputs
    claims_input = Input(shape=(max_length,), dtype='int32')
    evidences_input = Input(shape=(max_length,), dtype='int32')

    # Shared Embedding layer
    embedding_layer = Embedding(vocab_size, embedding_dim, weights=[embedding_matrix], trainable=False)

    # LSTM layers
    claims_embeddings = embedding_layer(claims_input)
    evidences_embeddings = embedding_layer(evidences_input)

    claims_lstm = LSTM(16)(claims_embeddings)
    evidences_lstm = LSTM(16)(evidences_embeddings)

    # Concatenate and output
    concatenated = concatenate([claims_lstm, evidences_lstm])
    concatenated = Dropout(0.5)(concatenated)
    output = Dense(1, activation='sigmoid')(concatenated)  # Binary output

    model = Model(inputs=[claims_input, evidences_input], outputs=output)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    return model

model = create_model()
print(model.summary())


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 278)]                0         []                            
                                                                                                  
 input_2 (InputLayer)        [(None, 278)]                0         []                            
                                                                                                  
 embedding (Embedding)       (None, 278, 50)              528650    ['input_1[0][0]',             
                                                                     'input_2[0][0]']             
                                                                                                  
 lstm (LSTM)                 (None, 16)                   4288      ['embedding[0][0]']       

###  Training the Model

In [8]:
# Mock labels for training (you need actual labels here)
labels = np.random.randint(2, size=len(claims))

# Train the model
model.fit([claims_padded, evidences_padded], labels, epochs=10, batch_size=32, validation_split=0.1)


Epoch 1/10
35/35 [==============================] - 2s 36ms/step - loss: 0.6934 - accuracy: 0.4796 - val_loss: 0.6931 - val_accuracy: 0.5041
Epoch 2/10
35/35 [==============================] - 1s 25ms/step - loss: 0.6934 - accuracy: 0.5122 - val_loss: 0.6931 - val_accuracy: 0.5041
Epoch 3/10
35/35 [==============================] - 1s 30ms/step - loss: 0.6935 - accuracy: 0.5014 - val_loss: 0.6931 - val_accuracy: 0.5041
Epoch 4/10
35/35 [==============================] - 1s 26ms/step - loss: 0.6932 - accuracy: 0.5050 - val_loss: 0.6931 - val_accuracy: 0.5041
Epoch 5/10
35/35 [==============================] - 1s 37ms/step - loss: 0.6929 - accuracy: 0.5113 - val_loss: 0.6932 - val_accuracy: 0.5041
Epoch 6/10
35/35 [==============================] - 1s 26ms/step - loss: 0.6929 - accuracy: 0.4977 - val_loss: 0.6931 - val_accuracy: 0.5041
Epoch 7/10
35/35 [==============================] - 1s 25ms/step - loss: 0.6933 - accuracy: 0.5059 - val_loss: 0.6931 - val_accuracy: 0.5041
Epoch 8/10
35

In [9]:


# Sample data
claim = "Climate change is primarily caused by carbon emissions."
evidences = [
    "Carbon emissions have significantly increased in the last century.",
    "Polar bear populations have decreased due to habitat loss.",
    "Renewable energy sources are becoming more economically viable."
]

# Assuming all text should be padded to the same length
max_length = 50  # This should be the same as used during model training

# Preprocess claim and evidences
claim_preprocessed = preprocess_text(claim, tokenizer, max_length)
evidence_preprocessed = [preprocess_text(ev, tokenizer, max_length) for ev in evidences]

# Predict the relevance of each evidence
results = []
for evidence in evidence_preprocessed:
    # Assuming the model expects a list of [claim, evidence]
    prediction = model.predict([claim_preprocessed, evidence])[0]
    results.append(prediction)

# Interpret and display results
related_evidence = [(ev, res) for ev, res in zip(evidences, results) if res > 0.5]
print("Related evidence to the claim:")
for ev, res in related_evidence:
    print(f"Evidence: {ev}, Score: {res}")

NameError: name 'preprocess_text' is not defined